1. Importación de Librerías

In [ ]:
import requests
import pandas as pd
import time
import random
from bs4 import BeautifulSoup
from datetime import datetime, timezone, timedelta

2. Declaración de variables

In [ ]:
# Objetivo: Hacker News
url = "https://news.ycombinator.com/"

# Listas para almacenar datos
titulos = []
links = []
puntajes = []
autores = []
comentarios = []
fechas = []

# Número de páginas a scrapear
num_pgs = 3

# Timestamp de extracción (GMT-5: EC)
ecuador_tz = timezone(timedelta(hours=-5))
timestamp_extraccion = datetime.now(ecuador_tz).strftime("%Y-%m-%d %H:%M:%S")

3. Petición HTTP

In [ ]:
for _ in range(num_pgs):
    response = requests.get(url) # Devuelve <Response [200]>: OK
    soup = BeautifulSoup(response.text, "html.parser")
    # Extracción de data
    items = soup.select(".athing")

    for item in items:
        titulo = item.select_one(".titleline a").text
        link = item.select_one(".titleline a")["href"]
        subtexto = item.find_next_sibling("tr").select_one(".subtext")
        puntaje = subtexto.select_one(".score")
        puntaje = puntaje.text if puntaje else "0 points"
        autor = subtexto.select_one(".hnuser")
        autor = autor.text if autor else "Sin autor"
        comentario = subtexto.select("a")[-1].text
        publicacion = subtexto.select_one(".age")
        fecha_publicacion = publicacion.text if publicacion else "Sin fecha"

        # Guardado en listas
        titulos.append(titulo)
        links.append(link)
        puntajes.append(puntaje)
        autores.append(autor)
        comentarios.append(comentario)
        fechas.append(fecha_publicacion)

    # Siguiente página
    siguiente = soup.select_one("a.morelink")
    if siguiente:
        url = "https://news.ycombinator.com/" + siguiente["href"]

    # Tiempo de espera (aleatorio)
    time.sleep(random.uniform(2, 5))

4. Creación de DataFrame

In [ ]:
df = pd.DataFrame({
    "titulo": titulos,
    "link": links,
    "puntaje": puntajes,
    "autor": autores,
    "comentarios": comentarios,
    "fechaPublicacion": fechas
})

In [ ]:
df.count()

In [ ]:
# Previsualización de data - 3 filas
display(df.head(3))

5. Limpieza de DF

In [ ]:
df["puntaje"] = df["puntaje"].str.replace(" points", "").astype(int)
df["comentarios"] = (
    df["comentarios"]
    .str.replace("\xa0", "") # Error por comentario
    .str.replace("comments", "")
    .str.replace("comment", "")
    .str.strip()
    .str.replace("discuss", "0") # Error por texto
    .str.replace("hide", "0") # Error por texto
    .astype(int)
)
df ["timestamp_extraccion"] = timestamp_extraccion

In [ ]:
media = df["puntaje"].mean()

def clasificar(puntaje):
    if puntaje >= media *1.2:
        return "Alto"
    elif puntaje >= media *0.8:
        return "Medio"
    else:
        return "Bajo"

df["nivelPuntaje"] = df["puntaje"].apply(clasificar)

In [ ]:
df = df.sort_values(by = "puntaje", ascending = False)

In [ ]:
display(df.head(3))

6. Generación CSV

In [ ]:
df.to_csv("dataset_hacker_news.csv", index = False)
print("Proceso completado, archivo generado.")
print(df.head(3))